## Problem: Save and Load Your PyTorch Model

### Problem Statement
You are tasked with saving a trained PyTorch model to a file and later loading it for inference or further training. This process allows you to persist the trained model and use it in different environments without retraining.

### Requirements
Implement four functions:

1. **`save_weights(model, path)`** — save the model's **state dictionary** (`model.state_dict()`) to `path` with `torch.save`.

2. **`load_for_inference(model_class, path)`** → model
   - `model_class` is a zero-argument callable that builds a fresh, untrained model (here the class `SimpleModel`).
   - Load the state dict with `torch.load(path, map_location="cpu", weights_only=True)`, apply it with `load_state_dict`, switch the model to **evaluation mode**, and return it.
   - The loaded model must produce exactly the same predictions as the original.

3. **`save_checkpoint(model, optimizer, epoch, path)`** — save one dict with the keys `"epoch"`, `"model_state_dict"` and `"optimizer_state_dict"`.

4. **`load_checkpoint(model, optimizer, path)`** → epoch
   - Restore the model's **and the optimizer's** state into the objects passed in, and return the stored epoch.
   - Continuing training from the restored model and optimizer must give exactly the same weights as continuing the original run.

### Notes
- Save `state_dict()` rather than `torch.save(model)`: pickling the whole module ties the file to your class's code and import path, while a state dict is just a mapping from names to tensors. It includes **buffers** such as BatchNorm's running statistics, which `named_parameters()` does not.
- `map_location` lets a checkpoint written on a GPU be loaded on a CPU-only machine (or onto a different device).
- Since PyTorch 2.6, `torch.load` defaults to `weights_only=True`, which refuses to unpickle arbitrary Python objects; state dicts and plain dicts of tensors and numbers still load. Keep it that way: unpickling a file you did not write can run arbitrary code.
- `model.eval()` switches layers such as dropout and batch norm to inference behaviour. `SimpleModel` has neither, but forgetting it is a classic bug (the grader's test model has both).
- Resuming training needs the optimizer state too — here SGD's momentum buffers (for Adam, the first and second moment estimates and the step count used for bias correction). Weights alone are not enough.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        return self.fc(x)

# Create and train the model
torch.manual_seed(42)
model = SimpleModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Training loop
X = torch.rand(100, 1)
y = 3 * X + 2 + torch.randn(100, 1) * 0.1
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, y)
    loss.backward()
    optimizer.step()

In [ ]:
def save_weights(model, path):
    """Save model.state_dict() to `path`."""
    torch.save(model.state_dict(), path)


def load_for_inference(model_class, path):
    """Build model_class(), load the state dict saved at `path` (map_location="cpu",
    weights_only=True), switch it to eval mode and return it."""
    model = model_class()
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.eval()
    return model


def save_checkpoint(model, optimizer, epoch, path):
    """Save {"epoch", "model_state_dict", "optimizer_state_dict"} to `path`."""
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, path)


def load_checkpoint(model, optimizer, path):
    """Restore the model and the optimizer (in place) from the checkpoint at `path`;
    return the stored epoch."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint["epoch"]

In [ ]:
# Save the trained weights, then load them into a fresh model for inference
save_weights(model, "model.pth")
loaded_model = load_for_inference(SimpleModel, "model.pth")

# Save a training checkpoint, then restore it into a fresh model and optimizer
save_checkpoint(model, optimizer, epochs, "checkpoint.pth")
resumed_model = SimpleModel()
resumed_optimizer = optim.SGD(resumed_model.parameters(), lr=0.01, momentum=0.9)
start_epoch = load_checkpoint(resumed_model, resumed_optimizer, "checkpoint.pth")
resumed_model.train()  # training would continue from start_epoch

In [ ]:
# Verify the model works after loading
X_test = torch.tensor([[0.5], [1.0], [1.5]])
with torch.no_grad():
    predictions = loaded_model(X_test)
    print(f"Predictions after loading: {predictions}")
    assert torch.equal(predictions, model(X_test)), "the loaded model disagrees with the original"
    assert torch.equal(resumed_model(X_test), model(X_test)), "the resumed model disagrees with the original"
assert not loaded_model.training, "the inference model should be in eval mode"

# The optimizer state came back too: SGD's momentum buffers match the original's
for p_orig, p_new in zip(model.parameters(), resumed_model.parameters()):
    assert torch.equal(optimizer.state[p_orig]["momentum_buffer"],
                       resumed_optimizer.state[p_new]["momentum_buffer"])
assert start_epoch == epochs

# So one more training step lands in exactly the same place
for m, opt in ((model, optimizer), (resumed_model, resumed_optimizer)):
    opt.zero_grad()
    criterion(m(X), y).backward()
    opt.step()
assert all(torch.equal(a, b) for a, b in zip(model.parameters(), resumed_model.parameters()))
print("Weights and optimizer state restored exactly.")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("save-model", save_weights=save_weights, load_for_inference=load_for_inference, save_checkpoint=save_checkpoint, load_checkpoint=load_checkpoint)